# 一键流程：PDF 文件夹 → 入库 → 发布 → 回答问题集

从上到下依次执行即可：对一个文件夹里的每份 PDF 依次做 ingest → requalify → qualify → index → publish → tree，
然后（可选）用同一套 `document-catalog` 链路在进程内回答一份问题集，并写出 `report.json` / `report.md`（固定写到项目根的 `data/reports/<题集文件名>/`）。
已处理过的 PDF 重跑只回放缓存，不再调用模型。

## 前置条件

1. 项目根 `.env` 按 `.env.example` 填好（`.env` 固定在项目根，与 notebook 的工作目录无关，不提交）：
   - LLM：`OPENAI_API_KEY` / `OPENAI_BASE_URL` / `OPENAI_MODEL`（别名 `APP_LLM_API_KEY` / `APP_LLM_BASE_URL` / `APP_LLM_MODEL`）；
   - Embedding：最简单是设 `OPENAI_EMBEDDING_MODEL`（与 LLM 同一个 `OPENAI_BASE_URL` 网关、同一个 key）；独立 loopback 服务才用 `APP_EMBEDDING_*`；
   - Rerank：`APP_RERANK_*`（本地模型，loopback 地址；没有网关回退）；
   - 输入位置：`NB_PDF_DIR`（必填，只读）、`NB_QUESTIONS_PATH`（别名 `DATASET_PATH`，可选）。输出位置不可配置：本 notebook **不使用** `NB_REPORT_DIR`（该变量只对 CLI / 直接调用 `run_folder_pipeline` 有效），见下面「运行产物位置」。
   - 口令加密的 PDF：`PDF_INGEST_PASSWORD`（不带 `APP_` 前缀；需要打开口令而它未设置或不对时，该 PDF 以 `failed` 结束、进度输出里直接写明原因；未加密的 PDF 不受影响）。
2. 用到本地模型（独立 embedding 服务或 rerank）时，**本地模型隧道要先起**（本流程自己不会启动它）：
   `.venv/bin/python scripts/enterprise_pdf_rag/local_model_tunnel.py start`
3. 用项目的 `.venv` 作为 kernel（已 `pip install -e .`）。

本 notebook 不含任何密钥，也不向 `os.environ` 写入配置；唯一的例外是末尾「可选：答案评测」一节在 import deepeval 之前设置的四个 `DEEPEVAL_*` 第三方库开关（非密钥）。

## 运行产物位置

所有运行产物都在项目根的 `data/` 下，**绝不写 PDF 源目录**（它按只读对待）；下列目录不存在时自动创建：

```text
<项目根>/data/
├── ingestion/                  # 入库产物：各文档的 source / processing store、model-cache、answers-audit.sqlite（APP_INGESTION_DIR 可改，但必须仍在 data/ 之内）
├── reports/<题集文件名>/        # report.json / report.md，每次运行都写（没有题集时目录名为 run-folder）
└── eval/<题集文件名>/           # 可选评测一节的输出（含 .deepeval/ 缓存）
```

配置 cell 之后有一格「写入目录护栏」：`APP_INGESTION_DIR`、报告目录、评测输出目录解析后只要不在 `data/` 之内、或落在 PDF 源目录之内（或就是它），就在任何写入发生之前抛异常，Run All 停在那里。

## 测试数据

默认的合成 PDF 目录 `data/fixtures/pdf` 被 `.gitignore` 的 `data/*` 忽略，不随仓库同步；机器上没有时用以下命令生成（需要 dev 依赖 reportlab）：

```bash
.venv/bin/python scripts/make_fixtures_pdf.py
```

## 运行时行为

- 缺少必需配置时会在花任何 LLM 预算之前抛 `PreflightError` 并指出缺哪些变量。
- 开跑前会向 embedding 服务探测一次（`embed_query("preflight")`），用来确认网关或隧道已通。
- 只有题集里有 `rerank: true` 的用例时才需要 `APP_RERANK_*` 配置。

In [ ]:
# 安装本仓库（editable）及一键流程需要的 extras：pdf = pdfspine，service = 进程内评测用的 fastapi/httpx。
# 已经装过的环境可以跳过这一格。Databricks 上执行完要重启解释器：dbutils.library.restartPython()
%pip install -q -e "..[pdf,service]"

In [ ]:
import os

# 只在 Databricks 上重启 Python（让上一格 %pip 装的包生效）；其它环境什么都不做
# 本格是"notebook 不直接读环境变量"规则的例外：它在 import ragspine 之前执行，拿不到 get_settings()，只能直接读环境变量做平台探测
if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "dbutils" in globals():
    dbutils.library.restartPython()  # noqa: F821 - dbutils 由 Databricks 注入
else:
    print("非 Databricks 环境，跳过")

In [ ]:
# editable 安装靠 .pth 把 <repo>/src 加进 sys.path；Databricks 上该 .pth 不生效时，
# site-packages/ragspine/ 只剩 _llms/（无 __init__.py），import ragspine 会命中这个
# namespace package，于是找不到 ragspine.common。这里显式把 src 放到 sys.path 最前
# （必须在 restart 之后）；.pth 已生效的环境里这只是无害的重复。
import sys
from pathlib import Path

_src = Path.cwd().parent / "src"
if (_src / "ragspine" / "__init__.py").is_file() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
print("src 路径 =", _src, "| 已在 sys.path:", str(_src) in sys.path)

In [ ]:
# 导入诊断：确认 import ragspine 实际命中的是哪份代码（restart 之后执行）
# - __file__ 为 None            → 命中了同名目录形成的 namespace package，看 __path__ 是谁在遮蔽（见上一格 src-path）
# - __file__ 在 site-packages 下 → 装的是 PyPI 发行版（0.17.2 及更早没有 common/evidence/configs.py）
# - __file__ 指向 .../src/ragspine/__init__.py → 上面的 editable 安装已生效
import importlib.metadata as md
import importlib.util
import sys

try:
    import ragspine
except ImportError as exc:
    ragspine = None
    print("import ragspine 失败 =", repr(exc))

if ragspine is not None:
    print("ragspine.__file__ =", ragspine.__file__)
    print("ragspine.__path__ =", list(getattr(ragspine, "__path__", [])))
    for mod in ("ragspine.common", "ragspine.common.evidence", "ragspine.common.evidence.configs"):
        try:
            found = importlib.util.find_spec(mod) is not None
        except ImportError:
            found = False
        print(f"{mod}: {'OK' if found else '找不到'}")
for dist in ("ragspine", "rag-spine"):
    try:
        print(f"{dist} 版本 =", md.version(dist))
    except md.PackageNotFoundError:
        print(f"{dist} 未安装")
print("sys.path 前 8 项 =", sys.path[:8])

In [ ]:
from ragspine.common.evidence.configs import get_settings
from ragspine.common.evidence.configs import ROOT_DIR

settings = get_settings()

# ---- 来自项目根 .env（或真实环境变量）：未设置时为 None ----
PDF_DIR = settings.pdf_source_dir  # NB_PDF_DIR
QUESTIONS = settings.questions_path  # NB_QUESTIONS_PATH：不设则只入库、不评测
# 报告目录固定为 <项目根>/data/reports/<题集文件名>/（没有题集时为 run-folder），每次运行都写 report.json / report.md；notebook 不使用报告目录的环境变量配置
REPORT_DIR = ROOT_DIR / "data" / "reports" / ("run-folder" if QUESTIONS is None else QUESTIONS.stem)
INGESTION_ROOT = settings.ingestion_root  # APP_INGESTION_DIR，默认 <APP_DATA_DIR>/ingestion = <项目根>/data/ingestion；必须在 data/ 之内（见下一格护栏）

# ---- 可在这里覆盖的调用参数 ----
MAX_LIVE_CALLS_PER_PDF = 100  # 每份 PDF 的 ingest 真实模型调用预算（0-200）；0 = 只回放缓存
MAX_LIVE_CALLS_TOTAL = None  # ingest + tree + 回答共用的总预算；None = 不设总预算
BUILD_TREE = True  # 发布后为每份文档建 document tree（路由用，会多花调用）
REQUALIFY = True  # 重新资格化图表等视觉对象（图表要靠它才进索引）

print("PDF 源目录 :", PDF_DIR)
print("题集       :", QUESTIONS)
print("报告目录   :", REPORT_DIR)
print("ingestion  :", INGESTION_ROOT)

In [ ]:
# ───────────── 写入目录护栏（必须在主运行之前）─────────────
# 本 notebook 的所有运行产物只许落在 <项目根>/data/ 下，且绝不写 PDF 源目录（它是只读的）。
# 任何一个写入目录不满足就在写入发生之前抛异常，让 Run All 停在这里；目录不存在由管线 / 评测自动创建。
from pathlib import Path

DATA_DIR = (ROOT_DIR / "data").resolve()


def check_write_dir(path, label, variable):
    """写入目录合规返回 None，否则返回简体中文原因（评测一节也用它：不合规时提示并跳过，不抛异常）。"""
    target = Path(path).expanduser().resolve()
    if not target.is_relative_to(DATA_DIR):
        return f"{label} {target}（由 {variable} 决定）不在项目根的 data/ 目录（{DATA_DIR}）之内：所有运行产物只许写到 data/ 下"
    if PDF_DIR is not None and target.is_relative_to(Path(PDF_DIR).expanduser().resolve()):
        return f"{label} {target}（由 {variable} 决定）位于 PDF 源目录 {Path(PDF_DIR).expanduser().resolve()}（NB_PDF_DIR）之内或就是它：PDF 目录是只读的，不能往里写任何东西"
    return None


for _label, _path, _variable in (
    ("ingestion 目录", INGESTION_ROOT, "APP_INGESTION_DIR / APP_DATA_DIR"),
    ("报告目录", REPORT_DIR, "notebook 固定的 data/reports/<题集文件名>"),
    ("评测输出目录", None if QUESTIONS is None else ROOT_DIR / "data" / "eval" / QUESTIONS.stem, "notebook 固定的 data/eval/<题集文件名>"),
):
    _problem = None if _path is None else check_write_dir(_path, _label, _variable)
    if _problem:
        raise RuntimeError("写入目录护栏拒绝运行（尚未写入任何文件）：" + _problem)
print("写入目录护栏通过：ingestion / 报告 / 评测输出都在", DATA_DIR, "之内，且不在 PDF 源目录之内")


In [ ]:
from enterprise_pdf_rag.adapters.folder_pipeline import run_folder_pipeline


def progress(event, payload):
    print(event, payload)


result = run_folder_pipeline(
    PDF_DIR,  # None → 取 NB_PDF_DIR；两处都没有会直接报错
    questions=QUESTIONS,
    ingestion_root=INGESTION_ROOT,
    max_live_calls_per_pdf=MAX_LIVE_CALLS_PER_PDF,
    max_live_calls_total=MAX_LIVE_CALLS_TOTAL,
    build_tree=BUILD_TREE,
    requalify=REQUALIFY,
    report_dir=REPORT_DIR,
    progress=progress,
)

In [ ]:
from pathlib import Path


def show(rows, columns):
    """不依赖 pandas 的等宽表格。"""
    cells = [[str(row.get(column, "")) for column in columns] for row in rows]
    widths = [
        max([len(column), *(len(line[i]) for line in cells)]) for i, column in enumerate(columns)
    ]
    print("  ".join(column.ljust(width) for column, width in zip(columns, widths)))
    for line in cells:
        print("  ".join(value.ljust(width) for value, width in zip(line, widths)))


print("ok              :", result.ok)
print("live_calls      :", result.live_calls.model_dump())
print("budget_exhausted:", result.budget_exhausted)
print("report_dir      :", result.report_dir)

documents = [
    {
        "pdf": Path(item.pdf_path).name,
        "status": item.status,
        "live_calls": item.live_calls,
        "elapsed_s": round(item.elapsed_s, 1),
        # 图表是否进了索引：qualification.chart_member_count > 0
        "chart_member_count": None
        if item.qualification is None
        else item.qualification.chart_member_count,
        "indexed_members": None if item.publication is None else item.publication.member_count,
        "error": item.error or "",
    }
    for item in result.documents
]
show(
    documents,
    ["pdf", "status", "live_calls", "elapsed_s", "chart_member_count", "indexed_members", "error"],
)

In [ ]:
if result.eval is None:
    print("没有题集（NB_QUESTIONS_PATH 未设置）：只做了入库。")
else:
    print("totals:", result.eval.totals)
    show(
        [
            {
                "case": case.case_id,
                "verdict": case.verdict,
                "status": case.status or "",
                "claims": case.claim_count,
                "cited_pages": list(case.cited_pages),
                "page_rank": case.page_rank,
                "ms": round(case.elapsed_ms),
                "failures": "; ".join(case.failures)[:80],
            }
            for case in result.eval.cases
        ],
        ["case", "verdict", "status", "claims", "cited_pages", "page_rank", "ms", "failures"],
    )

## 可选：单独拿一个回答模型客户端

`make_answer_llm()` 用 `.env` 里的 LLM 配置（`OPENAI_*`，别名 `APP_LLM_*`）构造 `JsonCompletionClient`，
缓存目录、调用预算、超时与种子取自 settings。构造本身不发请求；配置缺失时在这里抛 `ProviderConfigurationError`。

In [ ]:
from enterprise_pdf_rag.adapters.answer_llm import make_answer_llm

llm = make_answer_llm()  # 或 make_answer_llm(cache_dir=..., max_live_calls=...)
print(type(llm).__name__, "live calls so far:", llm.live_call_count)

## 可选：答案评测（deepeval，对照标准答案逐题判对错）

对上面 `result` 里回答过的题，用 deepeval 的 `GEval` 指标（`答案正确性`）让**裁判模型**对照题集里的标准答案（`expected`）逐题打分（0~1，`≥ PASS_THRESHOLD` 判为通过），给出准确率。判定交给裁判模型，不做规则比对（上面「运行时行为」里 `failures` 的子串 / 数字匹配只是轻量检查，没有准确率）。本节**可选**：没有题集、题集里没有任何 `expected`、没装 deepeval 时只打印提示并跳过，不影响上面的主流程。

- **输入**：优先用内存里的 `result`；`result` 不存在（只想重评已有产物）时读 `data/reports/<题集文件名>/report.json`（主流程每次运行都会写）。
- **标准答案**：`EvalCase` 里没有，按 `case_id` 到题集（`NB_QUESTIONS_PATH`，别名 `DATASET_PATH`，用 `ragspine.eval.retrieval_only.load_questions` 读）里取 `expected`；`id` 对不上时退回按问题原文匹配。
- **回答正文**：`EvalCase` / `report.json` 里没有模型回答的整段原文，只有 envelope 的 `claims`。送裁判的回答按优先级取：
  1. 问答审计库 `answers-audit.sqlite`（settings 的 `answer_audit_file`）里同一问题（有文档 sha 时同文档）最新一条的 `answer_text`，只读打开，找不到就跳过；
  2. 用已验证 `claims` 的 `text`（加 `value` / `unit`）拼成；
  3. 拒答时用 `拒答（abstain_reason）：abstain_detail`。

  采用的来源写在输出的 `answer_source` 列（`audit` / `claims` / `abstain`）。局限：按 `claims` 拼的回答只含通过校验的断言，不是用户最终看到的完整文字。
- **不送裁判**（标 `not_evaluated` 并写原因）：没有 `expected`；`verdict` 是 `http_error` / `routing_failed`（没问成）；没有任何回答内容。拒答但有 `expected` 的题照常送裁判（拒答对有具体答案的题算错）。
- **输出**：一律写到项目根下的 `data/eval/<题集文件名>/`（题集文件名取 `NB_QUESTIONS_PATH` 的 stem，`data/` 已被 git 忽略）：`eval_results.jsonl`（逐题，含 `eval_score` / `eval_pass` / `eval_reason` / `eval_status` / `judge_model` / `elapsed_s`，以及原样带上的 `verdict` / `answer_status` / `abstain_reason` / `claim_count` / `cited_pages` / `page_rank` / `answer_source` / `document_id`）、`eval_summary.md`、`eval.csv`（utf-8-sig）、`eval.xlsx`，deepeval 缓存 `.deepeval/` 也在这里。**本节绝不向报告目录 `data/reports/` 下写任何东西**（对它只读 `report.json`）；输出目录和实际生效的 deepeval 缓存目录（外部已预设 `DEEPEVAL_CACHE_FOLDER` 时以它为准）都必须位于 `data/` 之内、且不在 PDF 源目录之内，否则打印提示并整节跳过（输出目录落在报告目录之内同样跳过）。断点续跑：已评过且问题 / 期望 / 回答没变的题跳过，`error` 的重评，每题评完立刻追加写盘。
- **裁判模型**：用 `.env` 里的 LLM 配置（`OPENAI_API_KEY` / `OPENAI_BASE_URL` / `OPENAI_MODEL`，别名 `APP_LLM_*`），想换更强的模型改下面的 `JUDGE_MODEL`。

### 安装 deepeval（可选，默认不装）

deepeval 会把 `click` / `huggingface-hub` 降级，所以**不在项目依赖里，也不要装进本地 `.venv`**。

- **本地**：用 `uv run --with` 临时叠加（不改 `.venv` / `uv.lock`），再让 kernel 跑本 notebook，或命令行执行：
  `uv run --no-sync --with deepeval --with nbconvert --with ipykernel python -m nbconvert --to notebook --execute notebooks/run_folder.ipynb --output-dir <仓库外的临时目录>`。
- **Databricks**：取消下一个 cell 里 `%pip install` 那一行的注释并运行，然后重启 Python（再从头执行）。
- 没装 deepeval 时本节会提示并跳过。

deepeval 的四个开关（遥测 / 更新提示 / 不读 cwd 下的 `.env` / 本地状态目录）是非密钥的第三方库开关，只能写环境变量，本节在 import deepeval 之前设置（`os.environ.setdefault`）；除此之外本 notebook 不写任何环境变量。

In [ ]:
# 可选：Databricks 上取消下一行注释后运行（%pip 必须写在 cell 开头）。本地 .venv 不要运行，会降级 click / huggingface-hub；本地见上面的 uv run --with。
# %pip install -q deepeval

In [ ]:
# ───────────── 评测配置（本节所有可调项都在这一格）─────────────
import json
import os
import re
import sqlite3
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
from typing import Any

from ragspine.common.evidence.configs import ROOT_DIR
from ragspine.eval.retrieval_only import load_questions

# 评测用的标准字段 → 原始记录里的字段名。原始记录 = 一个 case（case_id / question / verdict ...）加上拼接的
# expected / answer / doc / error（见下一格 build_records）。字段名不同时改右边即可。
FIELD_MAP: dict[str, str] = {
    "id": "case_id",
    "question": "question",
    "expected": "expected",
    "answer": "answer",
    "doc": "doc",
    "error": "error",
    "status": "verdict",  # 有值且在 SKIP_STATUSES 里的题不送裁判
}
SKIP_STATUSES = {"http_error", "routing_failed"}  # 没问成的 verdict，不送裁判
# 原样带进输出（jsonl / csv / xlsx）的额外列；原始记录里没有的跳过
EXTRA_FIELDS = ["verdict", "answer_status", "abstain_reason", "claim_count", "cited_pages", "page_rank",
                "answer_source", "document_id"]

JUDGE_MODEL = ""  # 裁判模型名；空 = 用 settings.llm_model（OPENAI_MODEL / APP_LLM_MODEL）
PASS_THRESHOLD = 0.5  # GEval 得分（0~1）≥ 它判为通过
JUDGE_CONCURRENCY = 4  # 同时评测的题数
JUDGE_TIMEOUT = 120.0  # 单次裁判请求超时（秒）
JUDGE_TEMPERATURE: float | None = 0.0  # 模型不接受 temperature 时设 None（不传）
LLM_EXTRA_HEADERS: dict[str, str] = {}  # 网关要求的自定义请求头；默认空。不要把密钥明文写进 Notebook
JUDGE_FAKE = False  # 仅供自测：True 时用不联网的假裁判（期望答案里的数字 / 文字是否出现在回答里），走通整条 deepeval 链路

# 输出目录：一律是 <项目根>/data/eval/<题集文件名>/（评测实际执行就落盘）；绝不写进报告目录（它只用来读 report.json），也不写 PDF 源目录
EVAL_BLOCKED: str | None = None  # 非 None = 输出目录不合规，本节整体跳过（简体中文原因）
EVAL_DIR: Path | None = None if settings.questions_path is None else (ROOT_DIR / "data" / "eval" / settings.questions_path.stem).resolve()
if EVAL_DIR is not None and EVAL_DIR.is_relative_to(REPORT_DIR.resolve()):
    EVAL_BLOCKED = f"评测输出目录 {EVAL_DIR} 位于报告目录之内，已拒绝落盘并跳过评测；评测结果只许写到项目根的 data/eval/ 下"
    EVAL_DIR = None
elif EVAL_DIR is not None and (_eval_problem := check_write_dir(EVAL_DIR, "评测输出目录", "notebook 固定的 data/eval/<题集文件名>")):
    EVAL_BLOCKED = f"{_eval_problem}；已拒绝落盘并跳过评测"
    EVAL_DIR = None
EVAL_RESULTS = None if EVAL_DIR is None else EVAL_DIR / "eval_results.jsonl"
EVAL_SUMMARY = None if EVAL_DIR is None else EVAL_DIR / "eval_summary.md"
EVAL_CSV = None if EVAL_DIR is None else EVAL_DIR / "eval.csv"
EVAL_XLSX = None if EVAL_DIR is None else EVAL_DIR / "eval.xlsx"
print("评测输出目录:", EVAL_DIR or EVAL_BLOCKED or "（没有题集，不评测）")


In [ ]:
# ───────────── 读取回答、拼出标准答案与回答正文、筛出要评测的题 ─────────────
EVAL_SKIP: str | None = None  # 非 None = 本节整体跳过，值是原因（简体中文）


def load_audit_answers() -> tuple[dict[tuple[str, str], str], dict[str, str]]:
    """问答审计库里的回答原文：{(问题, 文档 sha): 文本} 与 {问题: 文本}（都取最新一条）。库不存在 / 打不开返回空。"""
    path = settings.answer_audit_file
    if not path.is_file():
        return {}, {}
    by_doc: dict[tuple[str, str], str] = {}
    by_question: dict[str, str] = {}
    try:
        conn = sqlite3.connect(f"{path.as_uri()}?mode=ro", uri=True)  # 只读打开，绝不写审计库
        try:
            for question, sha, text in conn.execute(
                "SELECT question, document_sha256, answer_text FROM answers "
                "WHERE answer_text IS NOT NULL AND answer_text <> '' AND error IS NULL ORDER BY id"
            ):
                by_doc[(question, sha)] = text
                by_question[question] = text
        finally:
            conn.close()
    except sqlite3.Error as exc:
        print(f"审计库读取失败（{type(exc).__name__}），回答改用 claims 拼接")
    return by_doc, by_question


def compose_answer(case: dict[str, Any], audit_doc: dict, audit_q: dict) -> tuple[str, str]:
    """(回答文本, 来源)：来源 audit / claims / abstain；什么都没有返回 ("", "none")。"""
    envelope = case.get("envelope") or {}
    question = case.get("question") or ""
    sha = envelope.get("document_sha256") or case.get("document_id") or ""
    text = audit_doc.get((question, sha)) or audit_q.get(question)
    if text:
        return text, "audit"
    if envelope.get("status") == "abstained" or case.get("status") == "abstained":
        reason = envelope.get("abstain_reason") or case.get("abstain_reason") or ""
        detail = envelope.get("abstain_detail") or ""
        return (f"拒答（{reason}）：{detail}".rstrip("：") if (reason or detail) else "拒答"), "abstain"
    parts = []
    for claim in envelope.get("claims") or ():
        line = claim.get("text") or ""
        value = claim.get("value")
        if value and str(value) not in line:
            line += f"（{value}{claim.get('unit') or ''}）"
        if line.strip():
            parts.append(line.strip())
    return "\n".join(parts), ("claims" if parts else "none")


def eval_cases() -> tuple[list[dict[str, Any]], str]:
    """(case 字典列表, 来源说明)：优先内存里的 result，没有就读报告目录下的 report.json。"""
    if "result" in globals() and getattr(result, "eval", None) is not None:
        return [case.model_dump() for case in result.eval.cases], "内存里的 result"
    if "result" in globals():
        raise LookupError("result.eval 为 None：运行时没有题集（NB_QUESTIONS_PATH 未设置）")
    report = REPORT_DIR / "report.json"
    if not report.is_file():
        raise LookupError(f"没有内存里的 result，也找不到 {report}：先运行上面的主流程生成报告")
    return ((json.loads(report.read_text(encoding="utf-8")).get("eval") or {}).get("cases") or []), str(report)


def build_records(cases: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """case 列表 → 原始记录列表（键见 FIELD_MAP 右边与 EXTRA_FIELDS）。"""
    questions = load_questions(settings.questions_path)
    by_id = {q.id: q for q in questions}
    by_text = {q.question.strip(): q for q in questions}
    audit_doc, audit_q = load_audit_answers()
    records = []
    for case in cases:
        q = by_id.get(case["case_id"]) or by_text.get((case.get("question") or "").strip())
        answer, source = compose_answer(case, audit_doc, audit_q)
        verdict = case.get("verdict")
        records.append({
            "case_id": case["case_id"], "question": case.get("question"),
            "expected": q.expected if q else None,
            "doc": "; ".join(q.doc) if q and q.doc else case.get("document_id"),
            "answer": answer, "answer_source": source,
            "error": f"{verdict}：{'；'.join(case.get('failures') or ())}" if verdict in SKIP_STATUSES else None,
            "verdict": verdict, "answer_status": case.get("status"), "abstain_reason": case.get("abstain_reason"),
            "claim_count": case.get("claim_count"), "cited_pages": case.get("cited_pages"),
            "page_rank": case.get("page_rank"), "document_id": case.get("document_id"),
        })
    return records


def normalize(raw: dict[str, Any]) -> dict[str, Any]:
    """按 FIELD_MAP 取标准字段，EXTRA_FIELDS 原样带上。"""
    rec = {std: raw.get(src) for std, src in FIELD_MAP.items()}
    rec["id"] = str(rec["id"])
    rec.update({f: raw[f] for f in EXTRA_FIELDS if f in raw})
    return rec


def skip_reason(r: dict[str, Any]) -> str | None:
    """不送裁判的原因；None = 要评测。"""
    if r.get("status") in SKIP_STATUSES:
        return f"没有回答：{r['status']}" + (f"（{r['error']}）" if r.get("error") else "")
    if r.get("error"):
        return f"回答出错：{r['error']}"
    if r.get("expected") in (None, ""):
        return "没有期望答案（expected）"
    if not str(r.get("answer") or "").strip():
        return "没有回答内容（无 claims 也不是拒答）"
    return None


def read_jsonl(path: Path | None) -> dict[str, dict[str, Any]]:
    """每个 id 的最后一条记录（保持首次出现的顺序）。"""
    records: dict[str, dict[str, Any]] = {}
    if path is not None and path.is_file():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line)
                records[str(rec.get("id"))] = rec
    return records


inputs: dict[str, dict[str, Any]] = {}
if EVAL_BLOCKED:
    EVAL_SKIP = EVAL_BLOCKED
elif settings.questions_path is None:
    EVAL_SKIP = "没有题集（NB_QUESTIONS_PATH / DATASET_PATH 未设置），没有标准答案，跳过评测"
else:
    try:
        _cases, _origin = eval_cases()
        for _raw in build_records(_cases):
            _rec = normalize(_raw)
            inputs[_rec["id"]] = _rec  # 同一 id 取最后一条
    except (LookupError, OSError, ValueError) as exc:  # 含题集解析错误 QuestionSetError（ValueError 子类）
        EVAL_SKIP = f"无法读取评测输入，跳过评测：{exc}"
    else:
        if not inputs:
            EVAL_SKIP = "没有可评测的题，跳过评测"
        elif not any(r.get("expected") not in (None, "") for r in inputs.values()):
            EVAL_SKIP = "题集里没有任何 expected（标准答案），跳过评测"

previous = read_jsonl(EVAL_RESULTS)
done = {k for k, e in previous.items() if e.get("eval_status") in ("evaluated", "not_evaluated") and k in inputs
        and all(e.get(f) == inputs[k].get(f) for f in ("question", "expected", "answer"))}
todo = [r for k, r in inputs.items() if k not in done]
if EVAL_SKIP:
    print(EVAL_SKIP)
else:
    n_skip = sum(1 for r in todo if skip_reason(r))
    print(f"输入：{_origin}；共 {len(inputs)} 题：已有结论 {len(done)}，本次处理 {len(todo)}（送裁判 {len(todo) - n_skip}，不评测 {n_skip}）")
    print("回答来源 :", dict(Counter(r.get("answer_source") for r in inputs.values())))
    print("不评测原因:", dict(Counter(skip_reason(r) or "（送裁判）" for r in inputs.values())))

### 裁判模型与评测指标

- `OpenAIJudge` 是 deepeval 的 `DeepEvalBaseLLM` 子类，包一个 OpenAI 兼容客户端（地址 / key / 模型名来自 settings 的 `llm_*`，带 `LLM_EXTRA_HEADERS`）。deepeval 调用 `generate(prompt, schema=<pydantic 模型>)` 要结构化输出：先用 `response_format={"type": "json_object"}`（端点不支持、返回 400 就去掉重试），再按 schema 解析；解析失败（如回复包在 ```json 里）就截取 `{...}` 再解析；还不行就返回原文交给 deepeval，仍失败则该题记 `error`，不中断整批。
- 指标是一个 `GEval`（`答案正确性`，`evaluation_params` = 问题 / 期望答案 / 实际回答），判定标准写在 `EVALUATION_STEPS` 里，可按需修改；裁判输出 0~10 分，deepeval 归一化到 0~1。
- deepeval 是**延迟 import**：下面的函数第一次被调用时才 import；`LLMTestCaseParams` 在 deepeval 4.x 改名为 `SingleTurnParams`，先导新名、失败回退旧名。不用 `deepeval.evaluate()`，每题新建一个 `GEval` 调 `measure()`（`async_mode=False`），并发由线程池负责。

In [ ]:
EVALUATION_STEPS = [
    "以期望答案为准，找出实际回答给出的最终结论（最终数值或最终表述），判断它与期望答案是否一致。",
    "数值允许单位换算与合理四舍五入：如 18.1 亿 = 1,810,000,000 = 1.81B，38% = 0.38；只要数值等价就算正确。",
    "实际回答里出现了正确数字、但最终结论是另一个数，或前后自相矛盾、给出多个互相冲突的答案，算错误。",
    "实际回答额外多说的内容（解释、出处、计算过程）不扣分，只要最终结论正确就算正确。",
    "实际回答明确表示无法找到、无法回答或信息不足，而期望答案有具体内容时，算错误。",
    "期望答案是文字（非数值）时，按语义是否一致判断，不要求字面一致。",
    "完全正确给 10 分，结论错误给 0 分；只有在部分正确（如多项答案只答对一部分）时才给中间分。理由用中文简要说明。",
]


def judge_settings() -> tuple[str, str, str]:
    """(模型名, base_url, key)：取自 settings（OPENAI_* 为主名，APP_LLM_* 为别名）；JUDGE_MODEL 可单独换模型名。"""
    llm_key = settings.llm_api_key.get_secret_value() if settings.llm_api_key else ""
    return JUDGE_MODEL or settings.llm_model or "", settings.llm_base_url or "", llm_key


def build_judge() -> tuple[Any, Any, Any]:
    """延迟 import deepeval，返回 (裁判实例, 每题新建指标的函数, LLMTestCase)；没装 deepeval 抛 ImportError。"""
    # deepeval 的四个开关是非密钥的第三方库开关，只认环境变量，必须在 import deepeval 之前设置
    os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "1")
    os.environ.setdefault("DEEPEVAL_UPDATE_WARNING_OPT_IN", "0")
    os.environ.setdefault("DEEPEVAL_DISABLE_DOTENV", "1")
    os.environ.setdefault("DEEPEVAL_CACHE_FOLDER", str(EVAL_DIR / ".deepeval"))
    # setdefault 不覆盖外部已设的值：取实际生效的缓存目录，同样必须在 data/ 之内且不在 PDF 源目录之内，否则跳过（在任何写入和 import deepeval 之前）
    cache_problem = check_write_dir(os.getenv("DEEPEVAL_CACHE_FOLDER") or EVAL_DIR / ".deepeval", "deepeval 缓存目录", "DEEPEVAL_CACHE_FOLDER")
    if cache_problem:
        raise RuntimeError(cache_problem)
    EVAL_DIR.mkdir(parents=True, exist_ok=True)

    from deepeval.metrics import GEval
    from deepeval.models import DeepEvalBaseLLM
    from deepeval.test_case import LLMTestCase

    try:
        from deepeval.test_case import SingleTurnParams as params  # deepeval 4.x 的新名字
    except ImportError:  # 旧版 deepeval
        from deepeval.test_case import LLMTestCaseParams as params
    import openai
    from pydantic import BaseModel, ValidationError

    def parse(text: str, schema: type[BaseModel] | None) -> Any:
        """按 schema 解析；失败就截取文本里的 {...} 再解析；还不行返回原文（deepeval 会自己再尝试解析）。"""
        if schema is None:
            return text
        start, end = text.find("{"), text.rfind("}")
        for candidate in (text, text[start:end + 1] if start != -1 and end > start else None):
            if candidate:
                try:
                    return schema.model_validate_json(candidate)
                except (ValidationError, ValueError):
                    pass
        return text

    class OpenAIJudge(DeepEvalBaseLLM):
        """OpenAI 兼容 Chat Completions 端点上的裁判模型。"""

        def __init__(self, model: str, base_url: str | None, llm_key: str | None):
            self.model_name = model
            self.client_args: dict[str, Any] = {"base_url": base_url or None, "api_key": llm_key or "EMPTY",
                                                "timeout": JUDGE_TIMEOUT, "default_headers": LLM_EXTRA_HEADERS or None}
            self.json_mode = True  # 端点不支持 response_format 时自动关掉
            self._async_client = None
            super().__init__(model)

        def load_model(self) -> Any:
            return openai.OpenAI(**self.client_args)

        def _kwargs(self, prompt: str, schema: type[BaseModel] | None) -> dict[str, Any]:
            kwargs: dict[str, Any] = {"model": self.model_name, "messages": [{"role": "user", "content": prompt}]}
            if JUDGE_TEMPERATURE is not None:
                kwargs["temperature"] = JUDGE_TEMPERATURE
            if schema is not None and self.json_mode:
                kwargs["response_format"] = {"type": "json_object"}
            return kwargs

        def generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
            kwargs = self._kwargs(prompt, schema)
            try:
                resp = self.model.chat.completions.create(**kwargs)
            except openai.BadRequestError:
                if "response_format" not in kwargs:
                    raise
                self.json_mode = False  # 端点不支持 JSON 模式：去掉重试，之后都不再带
                kwargs.pop("response_format")
                resp = self.model.chat.completions.create(**kwargs)
            return parse(resp.choices[0].message.content or "", schema)

        async def a_generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
            if self._async_client is None:
                self._async_client = openai.AsyncOpenAI(**self.client_args)
            kwargs = self._kwargs(prompt, schema)
            try:
                resp = await self._async_client.chat.completions.create(**kwargs)
            except openai.BadRequestError:
                if "response_format" not in kwargs:
                    raise
                self.json_mode = False
                kwargs.pop("response_format")
                resp = await self._async_client.chat.completions.create(**kwargs)
            return parse(resp.choices[0].message.content or "", schema)

        def get_model_name(self) -> str:
            return self.model_name

    class FakeJudge(DeepEvalBaseLLM):
        """仅供自测（JUDGE_FAKE=True）：不联网，按期望答案里的数字 / 文字是否出现在回答里给 9 分或 1 分。"""

        def __init__(self):
            super().__init__("fake-judge")

        def load_model(self) -> Any:
            return self

        @staticmethod
        def _section(prompt: str, title: str) -> str:
            m = re.search(rf"{title}:\n(.*?) \n\n", prompt, re.DOTALL)
            return m.group(1) if m else ""

        @staticmethod
        def _numbers(text: str) -> set[float]:
            return {float(n.replace(",", "")) for n in re.findall(r"\d[\d,]*(?:\.\d+)?", text)}

        def generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
            expected, actual = self._section(prompt, "Expected Output"), self._section(prompt, "Actual Output")
            wanted = self._numbers(expected)
            hit = wanted <= self._numbers(actual) if wanted else expected.strip().casefold() in actual.casefold()
            text = json.dumps({"reason": f"（假裁判）期望答案{'出现在' if hit else '没有出现在'}回答里", "score": 9 if hit else 1},
                              ensure_ascii=False)
            return parse(text, schema)

        async def a_generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
            return self.generate(prompt, schema)

        def get_model_name(self) -> str:
            return "fake-judge"

    if JUDGE_FAKE:
        judge: Any = FakeJudge()
    else:
        model, base_url, llm_key = judge_settings()
        if not model or not llm_key:
            raise RuntimeError("未配置裁判模型：请在 .env 设置 OPENAI_API_KEY / OPENAI_BASE_URL / OPENAI_MODEL（别名 APP_LLM_*），或改上面的 JUDGE_MODEL")
        judge = OpenAIJudge(model, base_url, llm_key)

    def make_metric() -> Any:
        """每题新建一个指标（GEval 实例会记录本次的 score / reason，不能在线程间共用）。"""
        return GEval(name="答案正确性", evaluation_steps=EVALUATION_STEPS,
                     evaluation_params=[params.INPUT, params.EXPECTED_OUTPUT, params.ACTUAL_OUTPUT],
                     threshold=PASS_THRESHOLD, model=judge, async_mode=False, verbose_mode=False)

    return judge, make_metric, LLMTestCase

### 逐题评测与汇总导出

`JUDGE_CONCURRENCY` 道题同时评测；每题完成立即追加写入 `eval_results.jsonl`，并打印 `[序号/总数] id 得分 通过/不通过 耗时`。单题出错（超时、裁判输出无法解析等）只记 `eval_status=error`，不中断整批，下次运行会重评。准确率 = 通过数 / 已评测数（不含未评测和报错的题）。
没装 deepeval、裁判模型没配置时打印提示并跳过（已有的 `not_evaluated` 结论仍会汇总）。

In [ ]:
eval_rows: list[dict[str, Any]] = []
if not EVAL_SKIP:
    to_judge = [r for r in todo if not skip_reason(r)]
    judge = make_metric = test_case_cls = None
    if to_judge:
        try:
            judge, make_metric, test_case_cls = build_judge()
        except ImportError:
            EVAL_SKIP = "没装 deepeval，跳过评测。安装方式见上面「安装 deepeval」（本地用 uv run --with deepeval，不要装进 .venv）"
        except RuntimeError as exc:
            EVAL_SKIP = str(exc) + "；跳过评测"
    if EVAL_SKIP:
        print(EVAL_SKIP)
    else:
        judge_name = judge.get_model_name() if judge else None
        lock = threading.Lock()
        finished = [0]

        def evaluate_one(r: dict[str, Any]) -> None:
            t0 = time.perf_counter()
            row = {"id": r["id"], "question": r.get("question"), "expected": r.get("expected"), "answer": r.get("answer"),
                   "doc": r.get("doc"), **{f: r.get(f) for f in EXTRA_FIELDS if f in r},
                   "eval_score": None, "eval_pass": None, "eval_reason": None,
                   "eval_status": "not_evaluated", "judge_model": None, "elapsed_s": 0.0}
            reason = skip_reason(r)
            if reason:
                row["eval_reason"] = reason
            else:
                row["judge_model"] = judge_name
                try:
                    metric = make_metric()
                    metric.measure(test_case_cls(input=str(r.get("question") or ""), actual_output=str(r.get("answer") or ""),
                                                 expected_output=str(r.get("expected"))), _show_indicator=False)
                    row.update(eval_score=round(float(metric.score), 4), eval_pass=bool(metric.success),
                               eval_reason=metric.reason, eval_status="evaluated")
                except Exception as exc:  # noqa: BLE001 - 单题失败只记下来，整批继续
                    row.update(eval_reason=f"{type(exc).__name__}: {exc}", eval_status="error")
                row["elapsed_s"] = round(time.perf_counter() - t0, 2)
            with lock:  # 每题完成立即追加写入
                EVAL_DIR.mkdir(parents=True, exist_ok=True)
                with EVAL_RESULTS.open("a", encoding="utf-8") as fh:
                    fh.write(json.dumps(row, ensure_ascii=False) + "\n")
                finished[0] += 1
                status = row["eval_status"]
                verdict = (f"{row['eval_score']:.2f} {'通过' if row['eval_pass'] else '不通过'}" if status == "evaluated" else
                           f"ERROR {row['eval_reason']}" if status == "error" else f"不评测：{row['eval_reason']}")
                print(f"[{finished[0]}/{len(todo)}] {row['id']}  {verdict}  {row['elapsed_s']:.1f}s", flush=True)

        t_start = time.time()
        if todo:
            with ThreadPoolExecutor(max_workers=max(1, JUDGE_CONCURRENCY)) as pool:
                list(pool.map(evaluate_one, todo))
        print(f"本次处理 {len(todo)} 题，用时 {time.time() - t_start:.1f}s")
        latest = read_jsonl(EVAL_RESULTS)
        eval_rows = [latest[k] for k in inputs if k in latest]

In [ ]:
if not eval_rows:
    if not EVAL_SKIP:  # 跳过的原因前面已经打印过
        print("没有评测结果")
else:
    import pandas as pd

    evaluated = [r for r in eval_rows if r["eval_status"] == "evaluated"]
    passed = sum(1 for r in evaluated if r["eval_pass"])
    n_not = sum(1 for r in eval_rows if r["eval_status"] == "not_evaluated")
    n_err = sum(1 for r in eval_rows if r["eval_status"] == "error")
    acc_text = f"{passed / len(evaluated):.1%}（{passed}/{len(evaluated)}）" if evaluated else "无（没有已评测的题）"
    judges = sorted({r["judge_model"] for r in evaluated if r.get("judge_model")})

    def _cell(text: Any, limit: int = 80) -> str:
        s = " ".join(str("" if text is None else text).split()).replace("|", "\\|")
        return s if len(s) <= limit else s[: limit - 1] + "…"

    lines = [
        "# 答案评测（deepeval GEval：答案正确性）", "",
        f"- 时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"- 裁判模型：{', '.join(judges) or '-'}；通过阈值 {PASS_THRESHOLD}",
        f"- **准确率：{acc_text}**",
        f"- 题数 {len(eval_rows)}：已评测 {len(evaluated)}，未评测 {n_not}，报错 {n_err}", "",
        "| ID | 问题 | 得分 | 通过 | 理由摘要 |", "|---|---|---|---|---|",
    ]
    for r in eval_rows:
        score_text = f"{r['eval_score']:.2f}" if r["eval_score"] is not None else "-"
        pass_text = {"evaluated": "✓" if r["eval_pass"] else "✗", "error": "报错", "not_evaluated": "未评测"}[r["eval_status"]]
        lines.append(f"| {_cell(r['id'], 40)} | {_cell(r['question'], 60)} | {score_text} | {pass_text} | {_cell(r['eval_reason'], 120)} |")

    def _flat(value: Any) -> Any:
        """列表 / 元组（如 cited_pages）转成逗号分隔文本，csv / xlsx 才写得进去。"""
        return ", ".join(map(str, value)) if isinstance(value, (list, tuple)) else value

    df = pd.DataFrame([{k: _flat(v) for k, v in r.items()} for r in eval_rows])
    written: list[Path] = []
    EVAL_SUMMARY.write_text("\n".join(lines) + "\n", encoding="utf-8")
    df.to_csv(EVAL_CSV, index=False, encoding="utf-8-sig")
    written += [EVAL_RESULTS, EVAL_SUMMARY, EVAL_CSV]
    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

        def clean_cell(v: Any) -> Any:
            return ILLEGAL_CHARACTERS_RE.sub("", v)[:32767] if isinstance(v, str) else v  # xlsx 不接受控制字符，单元格最长 32767 字符

        # DataFrame.map 需要 pandas>=2.1；更旧的版本退回 applymap
        (df.map(clean_cell) if hasattr(df, "map") else df.applymap(clean_cell)).to_excel(EVAL_XLSX, index=False)
        written.append(EVAL_XLSX)
    except ImportError:
        print("未安装 openpyxl，跳过 eval.xlsx")
    print(f"准确率 : {acc_text}；未评测 {n_not}，报错 {n_err}，共 {len(eval_rows)} 题")
    print("产出文件：" + "、".join(map(str, written)))
    display(df)